<a href="https://colab.research.google.com/github/EthanEShea/SmartVineMonitoring/blob/main/LeaveExtraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Turns the hard to look at 360 videos, into clear grouped frames of leaves.

In [ ]:
import cv2
import os

projectDir = "dataWine"
videoDir = os.path.join(projectDir, "Videos")
frameDir = os.path.join(projectDir, "Frames")
modelDir = os.path.join(projectDir, "models")
videoName = ""  # insert .mp4 here
videoPath = os.path.join(videoDir, videoName)

cap = cv2.VideoCapture(videoPath)
# Output path for the frames (update accordingly)
os.makedirs(frameDir, exist_ok=True)

# Creating frames from the video.
frameCount = 0
savedCount = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frameCount % 100 == 0: # adjust the frameCount as needed
        framePath = os.path.join(frameDir, f"walk_frame_{savedCount:05d}.jpg")
        cv2.imwrite(framePath, frame)
        print("Reading frame:", frameCount)
        savedCount += 1
    frameCount += 1

cap.release()
print("Frames processed:", frameCount)
print("Frames saved:", savedCount)

Used the images that were created from the video and used CVAT to create labeled data. Drew bounding boxes around leaves that were closer in the picture(to hopefully get a clearer picture). The model we trained for this worked for our videos, but more labeled images may be beneficial.

In [ ]:
data_yaml = f"""
path: {projectDir}
train: images/train
val: images/val

names:
  0: leafs
"""

with open(os.path.join(projectDir, "data.yaml"), "w") as f:
    f.write(data_yaml)

print("data.yaml created")

In [ ]:
!pip install ultralytics
!yolo detect train data=data.yaml model=yolov8n.pt epochs=50 imgsz=640

Runs the model, creates bounding boxes on the groups of leaves and then saves those bounding boxes to get a closer image of the leaves.

In [ ]:
# source is the same as outputPath (where our frames are)
# project is where the cropped images will go to
modelPath = os.path.join(modelDir, "best.pt")
resultsDir = os.path.join(projectDir, "yolo_results")
os.makedirs(resultsDir, exist_ok=True)

!yolo detect predict \
  model="{modelPath}" \
  source="{frameDir}" \
  save=True \
  save_crop=True \
  project="{resultsDir}" \
  name="predict_1" \
  exist_ok=True